# 1 · Azure SQL → Bronze
Load the 3 reference tables from Azure SQL into Bronze Delta tables (values unchanged, full overwrite each run).

| Azure SQL table | Bronze table | Rows | Key |
|---|---|---|---|
| machine_master | bronze_machine_master | 150 | machine_id |
| plant_master | bronze_plant_master | 159 | plant_id |
| failure_code_lookup | bronze_failure_code_lookup | 168 | failure_code |

## 1. Config

In [0]:

from pyspark.sql import functions as F

spark.sql("USE capstone_5iot.iot_schema")

SQL_SERVER = "capstone5dbserver.database.windows.net"
SQL_DB     = "capstone5"
SQL_SCHEMA = "iot_schema"          # Azure SQL schema that holds the 3 tables
SQL_USER   = dbutils.secrets.get("azure-sql-scope", "sql-user")   # Day 5 fix: user name from secret scope
SQL_PWD    = dbutils.secrets.get("azure-sql-scope", "sql-pwd")

JDBC_URL = f"jdbc:sqlserver://{SQL_SERVER}:1433;database={SQL_DB};encrypt=true;loginTimeout=30;"

# table -> (key, expected rows)
TABLES = {
    "machine_master":      ("machine_id",   150),
    "plant_master":        ("plant_id",     159),
    "failure_code_lookup": ("failure_code", 168),
}

def read_sql(query):
    return (spark.read.format("jdbc")
            .option("url", JDBC_URL)
            .option("query", query)
            .option("user", SQL_USER)
            .option("password", SQL_PWD)
            .load())

read_sql("SELECT 1 AS connection_ok").show()

## 2. Load to Bronze

In [0]:
for table in TABLES:
    (read_sql(f"SELECT * FROM dbo.{table}")
        .withColumn("_source_table", F.lit(f"azure_sql.{SQL_DB}.dbo.{table}"))
        .withColumn("_ingestion_ts", F.current_timestamp())
        .write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"bronze_{table}"))
    print(f"bronze_{table} loaded")

## 3. Validation
Source count = Bronze count = expected · key not null · key unique

In [0]:
results = []
for table, (key, expected) in TABLES.items():
    df      = spark.table(f"bronze_{table}")
    rows    = df.count()
    src     = read_sql(f"SELECT COUNT(*) AS c FROM dbo.{table}").first()["c"]
    nulls   = df.filter(F.col(key).isNull()).count()
    dupes   = rows - df.select(key).distinct().count()
    ok      = rows == src == expected and nulls == 0 and dupes == 0
    results.append((f"bronze_{table}", src, rows, expected, nulls, dupes, "PASS" if ok else "FAIL"))

res = spark.createDataFrame(results, "table string, source_rows long, bronze_rows long, expected long, key_nulls long, key_dupes long, status string")
display(res)
assert res.filter("status = 'FAIL'").count() == 0, "Validation failed"

In [0]:
%sql
SELECT * FROM bronze_machine_master LIMIT 5